# Introduction

This tutorial notebook contains four examples with different material systems to show new users how to use the `condevofm` (condevo for materials) package. Python scripts (for the execution on clusters etc.) with the same examples can be found in the respective folders. The results can be viewed with the `view_results()` function after the evolutions are completed. The following material systems are investigated here, where different features of the code are highlighted:

__1. Lennard-Jones (LJ) cluster --__ This example runs an evolution for a Lennard-Jones cluster with 13 atoms (LJ-13) with an `IndexConstrainer` that fixes or freezes single atoms by given index. The clusters are relaxed by the `FIRE` optimizer, parallelized on the CPU with the help of the `multiprocessing` package. We condition the sampling process with an `OriginCondition` that guides the particles to the origin of the Euclidean space to increase the probability of close-packed clusters and with a default `KNNNoveltyCondition` for higher diversity. The neural network is chosen as a simple `MLP`, which is used by the diffusion model `GGDDIM` class that incorporates gradient guidance with a `radial` geometry, again to bias the sampling in the direction of close-packed clusters. The evaluation function `evaluate_population_with_calc` is initialized with `get_potential_energy`, therefore the fitness of this evolutions is defined as the energy of the structures with a negative sign. This first example gives additional information to the respective configuration steps.

__2. Gold (Au) cluster --__ This example runs an evolution of a gold cluster with 13 atoms (Au-13). The same functionality as for the LJ-13 cluster is used to generate the randomized founder, but now with gold atoms by changing the `element` and `name` parameters in the `load_lj_cluster` function. We again utilize the `IndexConstrainer` and the `FIRE` optimizer, but now we need a MACE foundation model (in this case OMAT) to evaluate the clusters on the GPU. The conditioning, neural network und diffusion models are kept the same, but the gradient guidance geometry is now chosen as `ellipsoidal` to allow for wider two-dimensional clusters. The `steps` parameter was decreased significantly to reduce the computation time for presentation purposes, for useful results, a value around `steps=100` or higher is recommended.

__3. Gold sulfur (AuS) surface --__ This example runs an evolution of a gold sulfur surface. The dimensionality of the unit cell, the number of gold layers in the bulk and the number of gold and sulfur atoms on the top can be changed with the parameters of the `generate_aus_surface` function. Since we are now investigating a surface structure instead of a cluster, we want to constrain the free atoms onto a slim interval on top of the bulk material. We do this by changing three features. First, we use the `ThresholdConstrainer`, second we set the `AxisCondition` and third we choose the `axis` geometry for the gradient guidance of the `GGDDIM` class. The other parameters were chosen similar to the gold clusters. The `steps` parameter was decreased significantly to reduce the computation time for presentation purposes, for useful results, a value around `steps=100` or higher is recommended.

__4. Molybdenum disulfide (MoS2) monolayer --__ This example runs an evolution of a molybdenum disulfide monolayer. The founder is loaded from a structure file and contains three double sulfur vacancies in the center. Here we utilize the `SphereConstrainer` which fixes the atoms at the border of the unit cell to employ them as an anchor for the free atoms inside the defined sphere. The `axis` geometry for the gradient guidance is now enclosing the monolayer to keep the free atoms close to the two-dimensional material. The MACE model used in this example is not a foundation model, but a model trained with a data set containing only MoS2 monolayer structures. Otherwise the configuration is the same as for the gold sulfur surface. The `steps` parameter was decreased significantly to reduce the computation time for presentation purposes, for useful results, a value around `steps=100` or higher is recommended.


# Imports

First we import all necessary modules, classes and functions for the examples in this notebook.

In [ ]:
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=UserWarning)

import pathlib
import random
import shutil

import matplotlib.pyplot as plt
import numpy as np
import torch

from ase import Atoms
from ase.optimize import FIRE, BFGS
from ase.io.jsonio import encode
from ase.visualize.plot import plot_atoms

from mace.calculators import MACECalculator

from condevo.es.guidance import KNNNoveltyCondition

from condevofm.atoms import (
    IndexConstrainer, 
    ThresholdConstrainer, 
    SphereConstrainer,
    Optimizer, 
)
from condevofm.es import CHARLX
from condevofm.es.guidance import OriginCondition, AxisCondition
from condevofm.utils import CorrectedApplyLimitsObjective, run_evo, view_results
from ase.visualize import view

from clusters import (
    load_lj_cluster, 
    evaluate_lj_population_with_torch,
)

from surfaces import generate_aus_surface

from monolayers import load_mos2_monolayer

torch.set_default_dtype(torch.float64)

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED);


def plot_atoms_without_axes(atoms):
    fig, ax = plt.subplots()
    plot_atoms(atoms, ax=ax);
    ax.axis("off")


# Examples

## 1. Lennard-Jones (LJ) cluster

This example runs an evolution for a Lennard-Jones cluster with 13 atoms (LJ-13) with an `IndexConstrainer` that fixes or freezes single atoms by given index. The clusters are relaxed by the `FIRE` optimizer, parallelized on the CPU with the help of the `multiprocessing` package. We condition the sampling process with an `OriginCondition` that guides the particles to the origin of the Euclidean space to increase the probability of close-packed clusters and with a default `KNNNoveltyCondition` for higher diversity. The neural network is chosen as a simple `MLP`, which is used by the diffusion model `GGDDIM` class that incorporates gradient guidance with a `radial` geometry, again to bias the sampling in the direction of close-packed clusters. The evaluation function `evaluate_population_with_calc` is initialized with `get_potential_energy`, therefore the fitness of this evolutions is defined as the energy of the structures with a negative sign. This first example gives additional information to the respective configuration steps.

### Load founder structure

The coordinates for the global minimum are loaded from files saved in the directory `clusters/founders/`. Files for clusters with 13, 31 and 38 atoms are available, other files with clusters up to 150 atoms can be downloaded as a `.tar` file from `https://doye.chem.ox.ac.uk/jon/structures/LJ/tables.150.html`. After loading, additional information is shown: `max_span` (the maximum span of the structure used for conditioning), `dimensions` (number of atoms times three), `symbols` (chemical symbols for the `ase.atoms.Atoms` object) and `rep` (string representation for the destination path of the evolution). Also given is the energy of the structure, calculated by the `evaluate_lj_population_with_torch()` function, together with a visualiation of the cluster. For an interactive plot, comment in the last line `view(lj_atoms);`.

In [ ]:
N_ATOMS = 13 # 13, 31, 38

lj_cluster, max_span, dimensions, symbols, rep = load_lj_cluster(
    n_atoms=N_ATOMS,
    wales_path=pathlib.Path("clusters/founders"),
)

print(f"max_span = {max_span:.3f}")
print(f"dimensions = {dimensions}")
print(f"symbols = {symbols}")
print(f"rep = {rep}")

lj_cluster_pop = torch.from_numpy(lj_cluster.flatten())[None, :]
energy = evaluate_lj_population_with_torch(lj_cluster_pop)
print(f"Energy = {float(energy):.6f} eV")

lj_atoms = Atoms(symbols, lj_cluster)
plot_atoms_without_axes(lj_atoms)
### view(lj_atoms);


We always use randomized configurations as founder structures for the evolutions because we do not want to include any bias into the initialization of the algorithm. With the parameter `randomize=True` and the above chosen random seed we generate a randomized LJ-13 cluster. The same information as above is given. The energy of the founder structure is very high, since the randomization positioned some atoms very close to each other. For an interactive plot, comment in the last line `view(lj_atoms_random);`.

In [ ]:
lj_cluster_random, max_span, dimensions, symbols, rep = load_lj_cluster(
    n_atoms=N_ATOMS,
    wales_path=pathlib.Path("clusters/founders"),
    randomize=True,
    random_seed=SEED,
)

print(f"max_span = {max_span:.3f}")
print(f"dimensions = {dimensions}")
print(f"symbols = {symbols}")
print(f"rep = {rep}")

lj_cluster_random_pop = torch.from_numpy(lj_cluster_random.flatten())[None, :]
energy_random = evaluate_lj_population_with_torch(lj_cluster_random_pop)
print(f"Energy = {float(energy_random):.6f} eV")

lj_atoms_random = Atoms(
    symbols, np.reshape(lj_cluster_random, shape=(-1, 3))
)
plot_atoms_without_axes(lj_atoms_random);
### view(lj_atoms_random);


### Founder

We can now start with the configuration of the evolution. First, we define the randomized LJ-13 cluster from above as the `founder` structure for our evolution.

In [ ]:
# Initialize founder structure
lj_cluster_rand, max_span, dimensions, symbols, rep = load_lj_cluster(
    n_atoms=N_ATOMS,
    wales_path=pathlib.Path("clusters/founders"),
    randomize=True,
    random_seed=SEED,
)
founder = Atoms(symbols, np.reshape(lj_cluster_rand, shape=(-1, 3)))


### `Constrainer`

Next, we initialize the `Constrainer` class which defines if some atoms are fixed during the evolution or not. This can be changed by adding atom indices to the `fix_indices`. Since we want all atoms to be freely moveable, we use an empty list here. The printed output shows us that no atoms are fixed or freezed (i.e. atoms that are fixed during the sampling process but will be relaxed in every generation), while all 13 atoms are free. Other constrainer classes are available, for example for surfaces (see other examples below).

In [ ]:
# Initialize constraining parameters
constrainer = IndexConstrainer(
    fix_indices=[],
    freeze_indices=[],
)

print(constrainer.get_all_indices(founder))


### `Optimizer`

The `Optimizer` class defines the parameters for the relaxation step. Here, we chose with `calc="LJ"` the Lennard-Jones potential as evaluation method, which will be executed on the CPU (`device="cpu"`) in a parallelized fashion (`multiproc=True`) with 16 processes (`n_proc=16`) to decrease the computation time. The termination criteria for the relaxation is either a maximum force of 0.001 eV/Å (`fmax=0.001`) or 1000 steps performed (`steps=1000`). Additionally, we do not need a logfile (`logfile=None`) and want to obeserve the progress during the evolution (`progress_bar=True`). Important is the `e_cutoff` parameter which defines the energy threshold at which a penalty factor should be used. This is only necessary when the foundation model is assigning very low energies to unphysical structures. This value can change from founder to founder and should be set by the user independently. 

In [ ]:
# Initialize relaxing parameters
optimizer = Optimizer(
    founder_atoms=founder,
    constrainer=constrainer,
    calc="LJ",
    optimizer=FIRE,
    fmax=0.001,
    steps=1000,
    logfile=None,
    multiproc=True,
    n_proc=16,
    device="cpu",
    e_cutoff=-500,
    progress_bar=True,
)


### `CHARLX`

The evolutionary algorithm `CHARLX` itself is is a class derived from `CHARLES`. It is initialized, among other things, with the free positions of the founder structure `x0`, the population size `popsize` and the number of generations `n_gens`. For the conditioning we use two mechanisms, an `OriginCondition` to keep the atoms close to the origin and a `KNNNoveltyCondition` to increase the diversity of the samples. For a surface structure, the `AxisCondition` class can be used instead (see other examples below). The other parameters should only be changed if the user is sure it will improve the results. For further information on the parameters see the documentation of the `CHARLES` parent class.

In [ ]:
# Initialize condition around the origin
condition_obj = OriginCondition(
    n_atoms=optimizer.free_n_atoms,
    target=1.0,
    kwargs={"cond_threshold": max_span},
)

# Initialize evolutionary algorithm
es = CHARLX
es_config = dict(
    x0=optimizer.free_positions,
    constrainer=constrainer,
    optimizer=optimizer,
    conditions=(condition_obj, KNNNoveltyCondition()),
    popsize=16,
    n_gens=5,
    sigma_init=1.5,
    selection_pressure=20.0,
    elite_ratio=0.15,
    crossover_ratio=0.125,
    mutation_rate=0.05,
    diff_batch_size=256,
    diff_max_epoch=1000,
    buffer_size=1000,
    is_genetic_algorithm=True,
    adaptive_selection_pressure=True,
    readaptation=False,
    forget_best=False,
)


### Neural network

Now we initialize the neural network used inside the diffusion model `GGDDIM` for the predictions. We choose the default multi-layer-perceptron `MLP` and initialize the correct number of parameters and conditions based on the choices from above.

In [ ]:
# Initialize neural network
nn = "MLP"
nn_config = dict(
    num_hidden=96,
    num_layers=8,
    activation="LeakyReLU",
    num_params=optimizer.dimensions,
    num_conditions=len(es_config["conditions"]),
)


### Diffusion model

As diffusion model we use `GGDDIM`, the gradient guided version of the default implementation `DDIM`. As geometric guidance mode we choose the `radial` option, which defines in our case a sphere with radius `max_span * 1.5` positioned at the origin of the Euclidean space `diff_origin`. Futher information on the input parameters can be found both in the documentation of the `DDIM` and `GGDDIM` classes.

In [ ]:
# Initialize diffusion model
diff = "GGDDIM"
diff_config = dict(
    num_steps=5000,
    lamba_range=1.0,
    geometry="radial",
    axis=None,
    lower_threshold=0.0,
    upper_threshold=max_span*1.5,
    diff_origin=[0.0, 0.0, 0.0],
    overlap_penalty=True,
    train_on_penalty=True,
    progress_bar=True,
)


### Objective

Finally, we need to define an objective for the evolution. Given is the objective function `evaluate_population_with_calc()` which uses `ase` calculators to evaluate the energies of the sampled configurations. We maximize the fitness of the samples, which corresponds in our case to the lowest energy structures (`"evaluate_atoms": "get_potential_energy"`). The `obj_params` dictionary containes all necessary encoded `Optimizer` parameters, including `e_cutoff` for the evaluation step.

In [ ]:
# Initialize objective that will be maximized
obj = CorrectedApplyLimitsObjective(
    foo_module="condevofm.atoms",
    foo="evaluate_population_with_calc",
    foo_kwargs={
        "obj_params": optimizer.encode_params(),
        "evaluate_atoms": "get_potential_energy",
        "evaluate_kwargs": {},
    },
    maximize=True,
    dim=optimizer.dimensions,
)


### Run evolution

After all configurations are defined, we can execute the evolution. The `dst` variable determines in which folder the results should be saved. Here we use the `_evos` folder as destination and add further parameter choices to the name of the evolution folder for clarity. If a new evolution is started and a folder with the same name is already available, then this folder will be deleted. If this is not desired, comment out the line `shutil.rmtree(dst, ignore_errors=True)`. The function call `run_evo()` starts the evolution. The progress of the relaxation, evaluation and training steps are printed for every generation below, together with the maximum and average fitness values.

In [ ]:
# Define destination path for output data
dst = f"_evos/{rep}"
dst += f"_P-{es_config['popsize']}"
dst += f"_G-{es_config['n_gens']}"
dst += f"_F-{optimizer.fmax}"
dst += f"_S-{optimizer.steps}"
dst += f"_U-{diff_config['upper_threshold']:.3f}"

# Remove old folder before new evolution
shutil.rmtree(dst, ignore_errors=True)

# Execute CHARLX evolution
evo = run_evo(
    generations=es_config["n_gens"],
    es=es,
    es_config=es_config,
    nn=nn,
    nn_config=nn_config,
    diff=diff,
    diff_config=diff_config,
    objective=obj,
    dst=dst,
    params={"save_diffusion": False},
)


### View results

To view the energies of the sampled structures, execute the `view_results()` function. The energies for the best structure of every generation is plotted below, together with the energies of all structures in a specific generation `gen`, where the last generation is chosen as default. Also visualizations of the founder structure and the best configuration of the whole evolution are given. To interactively visualize these structures, set `show=True`.

In [ ]:
# Analyze output energies and structures
view_results(dst=dst, sort_samples=True, show=False)


## 2. Gold (Au) cluster

This example runs an evolution of a gold cluster with 13 atoms (Au-13). The same functionality as for the LJ-13 cluster is used to generate the randomized founder, but now with gold atoms by changing the `element` and `name` parameters in the `load_lj_cluster()` function. We again utilize the `IndexConstrainer` and the `FIRE` optimizer, but now we need a MACE foundation model (in this case OMAT) to evaluate the clusters on the GPU. The conditioning, neural network und diffusion models are kept the same, but the gradient guidance geometry is now chosen as `ellipsoidal` to allow for wider two-dimensional clusters. The `steps` parameter was decreased significantly to reduce the computation time for presentation purposes, for useful results, a value around `steps=100` is recommended. 

### Load founder structure

In [ ]:
au_cluster_rand, max_span, dimensions, symbols, rep = load_lj_cluster(
    n_atoms=13,
    wales_path=pathlib.Path("clusters/founders"),
    randomize=True,
    random_seed=SEED,
    element="Au",
    name="Au",
)

print(f"max_span = {max_span:.3f}")
print(f"dimensions = {dimensions}")
print(f"symbols = {symbols}")
print(f"rep = {rep}")

au_random_atoms = Atoms(symbols, np.reshape(au_cluster_rand, shape=(-1, 3)))
plot_atoms_without_axes(au_random_atoms);
### view(au_random_atoms);


### Configurate evolution

In [ ]:
# Initialize founder structure
au_cluster_rand, max_span, dimensions, symbols, rep = load_lj_cluster(
    n_atoms=13,
    wales_path=pathlib.Path("clusters/founders"),
    randomize=True,
    random_seed=SEED,
    element="Au",
    name="Au",
)
founder = Atoms(symbols, np.reshape(au_cluster_rand, shape=(-1, 3)))

# Initialize constraining parameters
constrainer = IndexConstrainer(
    fix_indices=[],
    freeze_indices=[],
)

# Initialize optimizing parameters
optimizer = Optimizer(
    founder_atoms=founder,
    constrainer=constrainer,
    calc="../models/mace-omat-pbe.model",
    optimizer=FIRE,
    fmax=0.001,
    steps=10, # 100,
    logfile=None,
    multiproc=False,
    n_proc=None,
    device="cuda",
    e_cutoff=-100,
    progress_bar=True,
)

# Initialize condition around the origin
condition_obj = OriginCondition(
    n_atoms=optimizer.free_n_atoms,
    target=1.0,
    kwargs={"cond_threshold": max_span},
)

# Initialize evolutionary algorithm
es = CHARLX
es_config = dict(
    x0=optimizer.free_positions,
    constrainer=constrainer,
    optimizer=optimizer,
    conditions=(condition_obj, KNNNoveltyCondition()),
    popsize=16,
    n_gens=5,
    sigma_init=1.5,
    selection_pressure=20.0,
    elite_ratio=0.15,
    crossover_ratio=0.125,
    mutation_rate=0.05,
    diff_batch_size=256,
    diff_max_epoch=1000,
    buffer_size=1000,
    is_genetic_algorithm=True,
    adaptive_selection_pressure=True,
    readaptation=False,
    forget_best=False,
)

# Initialize neural network
nn = "MLP"
nn_config = dict(
    num_hidden=96,
    num_layers=8,
    activation="LeakyReLU",
    num_params=optimizer.dimensions,
    num_conditions=len(es_config["conditions"]),
)

# Initialize diffusion model
diff = "GGDDIM"
diff_config = dict(
    num_steps=5000,
    lamba_range=1.0,
    geometry="ellipsoid",
    axis=None,
    lower_threshold=0.0,
    upper_threshold=list(max_span * np.array([1.5, 1.5, 3.0])),
    diff_origin=[0.0, 0.0, 0.0],
    overlap_penalty=True,
    train_on_penalty=True,
    progress_bar=True,
)

# Initialize objective that will be maximized
obj = CorrectedApplyLimitsObjective(
    foo_module="condevofm.atoms",
    foo="evaluate_population_with_calc",
    foo_kwargs={
        "obj_params": optimizer.encode_params(),
        "evaluate_atoms": "get_potential_energy",
        "evaluate_kwargs": {},
    },
    maximize=True,
    dim=optimizer.dimensions,
)


### Run evolution

In [ ]:
# Define destination path for output data
dst = f"_evos/{rep}"
dst += f"_P-{es_config['popsize']}"
dst += f"_G-{es_config['n_gens']}"
dst += f"_F-{optimizer.fmax}"
dst += f"_S-{optimizer.steps}"
dst += f"_U-{diff_config['upper_threshold'][0]:.3f}"
dst += f"-{diff_config['upper_threshold'][1]:.3f}"
dst += f"-{diff_config['upper_threshold'][2]:.3f}"
dst += f"_omat"

# Remove old folder before new evolution
shutil.rmtree(dst, ignore_errors=True)

# Execute CHARLX evolution
evo = run_evo(
    generations=es_config["n_gens"],
    es=es,
    es_config=es_config,
    nn=nn,
    nn_config=nn_config,
    diff=diff,
    diff_config=diff_config,
    objective=obj,
    dst=dst,
    params={"save_diffusion": False},
)


### View results

In [ ]:
# Analyze output energies and structures
view_results(dst=dst, sort_samples=True, show=False)


## 3. Gold sulfur (AuS) surface

This example runs an evolution of a gold sulfur surface. The dimensionality of the unit cell, the number of gold layers in the bulk and the number of gold and sulfur atoms on the top can be changed with the parameters of the `generate_aus_surface` function. Since we are now investigating a surface structure instead of a cluster, we want to constrain the free atoms onto a slim interval on top of the bulk material. We do this by changing three features. First, we use the `ThresholdConstrainer`, second we set the `AxisCondition` and third we choose the `axis` geometry for the gradient guidance of the `GGDDIM` class. The other parameters were chosen similar to the gold clusters. The `steps` parameter was decreased significantly to reduce the computation time for presentation purposes, for useful results, a value around `steps=100` or higher is recommended.

### Load founder structure

In [ ]:
# Initialize founder structure
founder, threshold, rep = generate_aus_surface(
    n_au_layers=4,
    n_au_atoms=4,
    n_s_atoms=4,
    repeat_x=3,
    repeat_y=3,
    transform=False, # True,
    vacuum=20.0,
    plot=True,
    show=False,
)

print(f"founder = {founder}")
print(f"threshold = {threshold:.3f}")
print(f"rep = {rep}")


### Configurate evolution

In [ ]:
# Initialize founder structure
founder, threshold, rep = generate_aus_surface(
    n_au_layers=4,
    n_au_atoms=4,
    n_s_atoms=4,
    repeat_x=3,
    repeat_y=3,
    transform=False, # True,
    vacuum=20.0,
)

# Initialize fixing parameters
constrainer = ThresholdConstrainer(
    fix_axis=[2],
    fix_threshold=7.25,
    freeze_axis=[2],
    freeze_threshold=6.25,
)

# Initialize relaxing parameters
optimizer = Optimizer(
    founder_atoms=founder,
    constrainer=constrainer,
    calc="../models/mace-omat-pbe.model",
    optimizer=FIRE,
    fmax=0.001,
    steps=10, # 100,
    logfile=None,
    multiproc=False,
    n_proc=None,
    device="cuda",
    e_cutoff=-1000,
    progress_bar=True,
)

# Initialize condition dependent on the chosen axis
axisCondition = AxisCondition(
    n_atoms=optimizer.free_n_atoms,
    target=1.0,
    kwargs={
        "cond_axis": [2],
        "cond_lower_threshold": 7.0,
        "cond_upper_threshold": 10.0,
    },
)

# Initialize evolutionary algorithm
es = CHARLX
es_config = dict(
    x0=optimizer.free_positions,
    constrainer=constrainer,
    optimizer=optimizer,
    conditions=(axisCondition, KNNNoveltyCondition()),
    popsize=16,
    n_gens=5,
    sigma_init=1.5,
    selection_pressure=20.0,
    elite_ratio=0.15,
    crossover_ratio=0.125,
    mutation_rate=0.05,
    diff_batch_size=256,
    diff_max_epoch=1000,
    buffer_size=1000,
    is_genetic_algorithm=True,
    adaptive_selection_pressure=True,
    readaptation=False,
    forget_best=False,
)

# Initialize neural network
nn = "MLP"
nn_config = dict(
    num_hidden=96,
    num_layers=8,
    activation="LeakyReLU",
    num_params=optimizer.dimensions,
    num_conditions=len(es_config["conditions"]),
)

# Initialize diffusion model
diff = "GGDDIM"
diff_config = dict(
    num_steps=5000,
    lamba_range=1.0,
    geometry="axis",
    axis=2,
    lower_threshold=7.0,
    upper_threshold=10.0,
    diff_origin=[0.0, 0.0, 8.5],
    overlap_penalty=True,
    train_on_penalty=True,
    progress_bar=True,
)

# Initialize objective that will be maximized
obj = CorrectedApplyLimitsObjective(
    foo_module="condevofm.atoms",
    foo="evaluate_population_with_calc",
    foo_kwargs={
        "obj_params": optimizer.encode_params(),
        "evaluate_atoms": "get_potential_energy",
        "evaluate_kwargs": {},
    },
    maximize=True,
    dim=optimizer.dimensions,
)


### Run evolution

In [ ]:
# Define destination path for output data
dst = f"_evos/{rep}"
dst += f"_P-{es_config['popsize']}"
dst += f"_G-{es_config['n_gens']}"
dst += f"_F-{optimizer.fmax}"
dst += f"_S-{optimizer.steps}"
dst += f"_L-{diff_config['lower_threshold']:.3f}"
dst += f"_U-{diff_config['upper_threshold']:.3f}"
dst += f"_O-{diff_config['diff_origin'][2]:.3f}"
dst += f"_omat"

# Remove old folder before new evolution
shutil.rmtree(dst, ignore_errors=True)

# Execute CHARLX evolution
evo = run_evo(
    generations=es_config["n_gens"],
    es=es,
    es_config=es_config,
    nn=nn,
    nn_config=nn_config,
    diff=diff,
    diff_config=diff_config,
    objective=obj,
    dst=dst,
    params={"save_diffusion": False},
)


### View results

In [ ]:
# Analyze output energies and structures
view_results(dst=dst, sort_samples=True, show=False)


## 4. Molybdenum disulfide (MoS2) monolayer

This example runs an evolution of a molybdenum disulfide monolayer. The founder is loaded from a structure file and contains three double sulfur vacancies in the center. Here we utilize the `SphereConstrainer` which fixes the atoms at the border of the unit cell to employ them as an anchor for the free atoms inside the defined sphere. The `axis` geometry for the gradient guidance is now enclosing the monolayer to keep the free atoms close to the two-dimensional material. The MACE model used in this example is not a foundation model, but a model trained with a data set containing only MoS2 monolayer structures. Otherwise the configuration is the same as for the gold sulfur surface. The `steps` parameter was decreased significantly to reduce the computation time for presentation purposes, for useful results, a value around `steps=100` or higher is recommended.

### Load founder structure

In [ ]:
# Initialize founder structure
founder, dimensions, symbols, rep, center, radius = load_mos2_monolayer(
    structure_path="monolayers/founders/MoS2_5x5x1_3v-s2.vasp", 
    plot=True,
    show=False,
)

print(f"founder = {founder}")
print(f"dimensions = {dimensions}")
print(f"symbols = {symbols}")
print(f"rep = {rep}")
print(f"center = {center}")
print(f"radius = {radius}")


### Configurate evolution

In [ ]:
# Initialize founder structure
founder, dimensions, symbols, rep, center, radius = load_mos2_monolayer(
    structure_path="monolayers/founders/MoS2_5x5x1_3v-s2.vasp", 
)

# Initialize fixing parameters
constrainer = SphereConstrainer(
    fix_center=center,
    fix_radius=radius,
    freeze_center=None,
    freeze_radius=None,
)

# Initialize relaxing parameters
optimizer = Optimizer(
    founder_atoms=founder,
    constrainer=constrainer,
    calc="../models/mace-mos2-swa.model",
    optimizer=FIRE,
    fmax=0.001,
    steps=10, # 100,
    logfile=None,
    multiproc=False,
    n_proc=None,
    device="cuda",
    e_cutoff=-510,
    progress_bar=True,
)

# Initialize condition dependent on the chosen axis
axisCondition = AxisCondition(
    n_atoms=optimizer.free_n_atoms,
    target=1.0,
    kwargs={
        "cond_axis": [2],
        "cond_lower_threshold": 10.0,
        "cond_upper_threshold": 14.0,
    },
)

# Initialize evolutionary algorithm
es = CHARLX
es_config = dict(
    x0=optimizer.free_positions,
    constrainer=constrainer,
    optimizer=optimizer,
    conditions=(axisCondition, KNNNoveltyCondition()),
    popsize=16,
    n_gens=5,
    sigma_init=1.5,
    selection_pressure=20.0,
    elite_ratio=0.15,
    crossover_ratio=0.125,
    mutation_rate=0.05,
    diff_batch_size=256,
    diff_max_epoch=1000,
    buffer_size=1000,
    is_genetic_algorithm=True,
    adaptive_selection_pressure=True,
    readaptation=False,
    forget_best=False,
)

# Initialize neural network
nn = "MLP"
nn_config = dict(
    num_hidden=96,
    num_layers=8,
    activation="LeakyReLU",
    num_params=optimizer.dimensions,
    num_conditions=len(es_config["conditions"]),
)

# Initialize diffusion model
diff = "GGDDIM"
diff_config = dict(
    num_steps=5000,
    lamba_range=1.0,
    geometry="axis",
    axis=2,
    lower_threshold=10.0,
    upper_threshold=14.0,
    diff_origin=[0.0, 0.0, 12.0],
    overlap_penalty=True,
    train_on_penalty=True,
    progress_bar=True,
)

# Initialize objective that will be maximized
obj = CorrectedApplyLimitsObjective(
    foo_module="condevofm.atoms",
    foo="evaluate_population_with_calc",
    foo_kwargs={
        "obj_params": optimizer.encode_params(),
        "evaluate_atoms": "get_potential_energy",
        "evaluate_kwargs": {},
    },
    maximize=True,
    dim=optimizer.dimensions,
)


### Run evolution

In [ ]:
# Define destination path for output data
dst = f"_evos/{rep}"
dst += f"_P-{es_config['popsize']}"
dst += f"_G-{es_config['n_gens']}"
dst += f"_F-{optimizer.fmax}"
dst += f"_S-{optimizer.steps}"
dst += f"_L-{diff_config['lower_threshold']:.3f}"
dst += f"_U-{diff_config['upper_threshold']:.3f}"
dst += f"_O-{diff_config['diff_origin'][2]:.3f}"
dst += f"_mace"

# Remove old folder before new evolution
shutil.rmtree(dst, ignore_errors=True)

# Execute CHARLX evolution
evo = run_evo(
    generations=es_config["n_gens"],
    es=es,
    es_config=es_config,
    nn=nn,
    nn_config=nn_config,
    diff=diff,
    diff_config=diff_config,
    objective=obj,
    dst=dst,
    params={"save_diffusion": False},
)


### View results

In [ ]:
# Analyze output energies and structures
view_results(dst=dst, sort_samples=True, show=False)
